# Lab 08: Text Preprocessing, Tokenization & Bag-of-Words / TF-IDF Classification

Welcome to Laboratory 08! In this lab, we enter **Natural Language Processing (NLP)**:
1. **Text Normalization & Cleaning**: Preprocess unstructured text using Regular Expressions (regex), lowercasing, and whitespace tokenization.
2. **TF-IDF Feature Representation**: Understand Term Frequency-Inverse Document Frequency (TF-IDF) weighting mathematics.
3. **Neural Sentiment Classification**: Train a PyTorch neural classifier on vectorized text representations.


## 1. Technical Preliminaries & Imports


In [ ]:
# Import standard libraries for regex processing, vectorization, and PyTorch
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer

# Seed random state
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Active Device:', device)


## 2. Text Normalization & TF-IDF Extraction

### Mathematical Formulation: TF-IDF Weighting
For a term $t$ in a document $d$ within a corpus $D$ of $N$ documents:
* **Term Frequency**: $\text{TF}(t, d) = \frac{f_{t, d}}{\sum_{t' \in d} f_{t', d}}$
* **Inverse Document Frequency**: $\text{IDF}(t, D) = \log\left(\frac{1 + N}{1 + |\{d \in D : t \in d\}|}\right) + 1$
* **TF-IDF Weight**: $\text{TF-IDF}(t, d, D) = \text{TF}(t, d) \times \text{IDF}(t, D)$
High values indicate words that are frequent in the specific document but rare across the corpus.


### Helper Function: `clean_text`
The `clean_text` function strips special punctuation, lowercases strings, and removes redundant whitespace.


In [ ]:
def clean_text(raw_text: str) -> str:
    """Normalizes a string by converting to lowercase and stripping non-alphabetical characters.
    
    Args:
        raw_text: Raw input sentence string
    Returns:
        Cleaned, normalized string containing only lowercased letters and spaces.
    """
    # Step 1: Convert to lower case
    text_lower = raw_text.lower()
    
    # Step 2: Strip all characters that are not lowercase ASCII letters or whitespace
    text_cleaned = re.sub(r'[^a-z\s]', '', text_lower)
    
    # Step 3: Remove redundant whitespace
    return text_cleaned.strip()

# Sample sentiment analysis corpus with binary labels (1=Positive, 0=Negative)
raw_corpus = [
    'I loved this movie! The acting was incredible.',
    'Terrible film, complete waste of time.',
    'A masterpiece of modern cinema! Highly recommended.',
    'Not worth your time. The script makes no sense.'
]
sentiment_labels = [1, 0, 1, 0]

# Clean all sentences in the corpus
cleaned_corpus = [clean_text(doc) for doc in raw_corpus]
for original, cleaned in zip(raw_corpus, cleaned_corpus):
    print(f'Original: "{original}" -> Cleaned: "{cleaned}"')

# Fit TF-IDF Vectorizer to extract numerical feature matrix
tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(cleaned_corpus).toarray()

print(f'\nTF-IDF Matrix Shape: {X_tfidf.shape} (4 Documents x {X_tfidf.shape[1]} Vocabulary Words)')
print(f'Extracted Vocabulary: {tfidf_vectorizer.get_feature_names_out()}')


## 3. Neural Sentiment Classification Architecture

### Architecture Overview: `TextSentimentClassifier`
The neural sentiment classifier processes TF-IDF vectors through dense layers:
* **Input Layer**: `Linear(V, 16)` where $V$ is vocabulary size.
* **Activation**: `ReLU()`.
* **Classification Output**: `Linear(16, 2)` producing binary sentiment logits (Negative vs. Positive).


In [ ]:
# Define the Neural Sentiment Classifier Architecture
class TextSentimentClassifier(nn.Module):
    """Multi-layer Perceptron for sentiment classification from TF-IDF vectors."""
    def __init__(self, vocab_size: int, hidden_dim: int = 16, num_classes: int = 2):
        super(TextSentimentClassifier, self).__init__()
        self.classifier = nn.Sequential(
            nn.Linear(vocab_size, hidden_dim), # Input projection: vocab_size -> hidden_dim
            nn.ReLU(),                         # Non-linear activation
            nn.Linear(hidden_dim, num_classes) # Output logits: hidden_dim -> 2 (Neg/Pos)
        )
        
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(x)

# Convert TF-IDF numpy array and labels to PyTorch Tensors
X_tensor = torch.tensor(X_tfidf, dtype=torch.float32).to(device)
y_tensor = torch.tensor(sentiment_labels, dtype=torch.long).to(device)

# Instantiate model, loss criterion, and optimizer
sentiment_model = TextSentimentClassifier(vocab_size=X_tfidf.shape[1], hidden_dim=16, num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(sentiment_model.parameters(), lr=0.05)

# Execute training loop over 30 epochs
for epoch in range(30):
    sentiment_model.train()
    
    # 1. Forward Pass
    logits = sentiment_model(X_tensor)
    loss = criterion(logits, y_tensor)
    
    # 2. Backward Pass & Parameter Updates
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Evaluate prediction accuracy
sentiment_model.eval()
with torch.no_grad():
    test_logits = sentiment_model(X_tensor)
    predictions = test_logits.argmax(dim=1)
    acc = (predictions == y_tensor).float().mean() * 100.0

print(f'Final Training Sentiment Classification Accuracy: {acc.item():.2f}%')
for doc, pred, true_lbl in zip(raw_corpus, predictions.cpu().numpy(), sentiment_labels):
    print(f'Text: "{doc}" | Predicted: {"Positive" if pred==1 else "Negative"} | Ground Truth: {"Positive" if true_lbl==1 else "Negative"}')


## 4. Summary & Key Takeaways
1. **Text Normalization**: Regex-based tokenization and stopword management establish standard clean inputs.
2. **TF-IDF**: Weighs terms based on specificity, suppressing high-frequency generic terms while amplifying discriminative keywords.
3. **Bag-of-Words Limitations**: Ignores word order and grammar ($n$-grams partially mitigate this, but recurrent and attention models provide full sequence understanding).
